# LLM-Roofline: Automated GPU Bottleneck Analyzer for LLM Inference

## Final end-to-end Colab project

**Goal:** Given a supported Hugging Face decoder-only LLM, an NVIDIA GPU, and an inference workload, automatically determine whether **prefill** and **decode** are primarily limited by GPU compute, memory traffic, or under-utilization; report the important performance statistics; recommend an optimization; and experimentally test whether that recommendation helps.

This is **not LLM training**. Qwen3 is our controlled benchmark family for building and validating the analyzer.

### Research questions

1. Under what conditions does LLM inference move from memory-dominated to compute-dominated?
2. How do model size, context length, batch size, inference phase, and numerical precision affect that transition?
3. Can a hardware-normalized Roofline metric correctly diagnose the bottleneck?
4. Can the diagnosis predict which optimization is worth trying?
5. Does the analyzer generalize beyond the Qwen family to other supported decoder-only Transformers?

### End-to-end pipeline

`CUDA GPU characterization -> Qwen scaling study -> bottleneck metric -> quantization validation -> cross-model validation -> arbitrary-model analyzer -> recommendations -> final interactive report`

> In Colab, enable an NVIDIA GPU first: **Runtime -> Change runtime type -> GPU**. A T4 is sufficient for the main experiments.

In [ ]:
!pip -q install -U "transformers>=4.51" accelerate sentencepiece pandas matplotlib ipywidgets


In [ ]:
import os
import gc
import math
import json
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "No CUDA GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU"

GPU_NAME = torch.cuda.get_device_name(0)
GPU_PROPS = torch.cuda.get_device_properties(0)

print("GPU:", GPU_NAME)
print("CUDA runtime used by PyTorch:", torch.version.cuda)
print("Compute capability:", f"{GPU_PROPS.major}.{GPU_PROPS.minor}")
print("VRAM:", round(GPU_PROPS.total_memory / 1024**3, 2), "GiB")
print()

!nvidia-smi
!nvcc --version

## 1. CUDA C++ microbenchmark

The custom CUDA kernel below measures practical global-memory copy bandwidth.

We count **both the read and the write**, so:

\[
Bandwidth = \frac{2 \times bytes \times iterations}{time}
\]

A second CUDA kernel is included as a scalar-FMA sanity check, but the main compute ceiling for the LLM Roofline model is measured later using large FP16 GEMMs because LLM matrix multiplications use Tensor Cores / cuBLAS rather than a scalar FMA loop.

In [ ]:
%%writefile gpu_microbench.cu
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CHECK_CUDA(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(1); \
    } \
} while (0)

__global__ void copy_kernel(const float4* __restrict__ in,
                            float4* __restrict__ out,
                            size_t n4) {
    size_t i = blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = blockDim.x * gridDim.x;
    for (; i < n4; i += stride) {
        out[i] = in[i];
    }
}

__global__ void fma_kernel(const float* __restrict__ in,
                           float* __restrict__ out,
                           size_t n,
                           int loops) {
    size_t i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;

    float x = in[i];
    float a0 = x + 0.01f;
    float a1 = x + 0.02f;
    float a2 = x + 0.03f;
    float a3 = x + 0.04f;
    float a4 = x + 0.05f;
    float a5 = x + 0.06f;
    float a6 = x + 0.07f;
    float a7 = x + 0.08f;

    #pragma unroll 4
    for (int j = 0; j < loops; ++j) {
        a0 = fmaf(a0, 1.0000001f, 0.0000001f);
        a1 = fmaf(a1, 1.0000002f, 0.0000002f);
        a2 = fmaf(a2, 1.0000003f, 0.0000003f);
        a3 = fmaf(a3, 1.0000004f, 0.0000004f);
        a4 = fmaf(a4, 1.0000005f, 0.0000005f);
        a5 = fmaf(a5, 1.0000006f, 0.0000006f);
        a6 = fmaf(a6, 1.0000007f, 0.0000007f);
        a7 = fmaf(a7, 1.0000008f, 0.0000008f);
    }

    out[i] = a0+a1+a2+a3+a4+a5+a6+a7;
}

int main() {
    cudaDeviceProp prop;
    CHECK_CUDA(cudaGetDeviceProperties(&prop, 0));
    printf("GPU_NAME=%s\n", prop.name);

    // ---------- Memory bandwidth ----------
    // 256 MiB input + 256 MiB output; safe on a 16 GiB T4.
    const size_t bytes = 256ULL * 1024ULL * 1024ULL;
    const size_t n = bytes / sizeof(float);
    const size_t n4 = n / 4;

    float *d_in = nullptr, *d_out = nullptr;
    CHECK_CUDA(cudaMalloc(&d_in, bytes));
    CHECK_CUDA(cudaMalloc(&d_out, bytes));
    CHECK_CUDA(cudaMemset(d_in, 1, bytes));

    int threads = 256;
    int blocks = prop.multiProcessorCount * 32;

    for (int i = 0; i < 10; ++i) {
        copy_kernel<<<blocks, threads>>>(
            reinterpret_cast<const float4*>(d_in),
            reinterpret_cast<float4*>(d_out),
            n4
        );
    }
    CHECK_CUDA(cudaDeviceSynchronize());

    cudaEvent_t start, stop;
    CHECK_CUDA(cudaEventCreate(&start));
    CHECK_CUDA(cudaEventCreate(&stop));

    const int copy_iters = 50;
    CHECK_CUDA(cudaEventRecord(start));
    for (int i = 0; i < copy_iters; ++i) {
        copy_kernel<<<blocks, threads>>>(
            reinterpret_cast<const float4*>(d_in),
            reinterpret_cast<float4*>(d_out),
            n4
        );
    }
    CHECK_CUDA(cudaEventRecord(stop));
    CHECK_CUDA(cudaEventSynchronize(stop));

    float copy_ms = 0.0f;
    CHECK_CUDA(cudaEventElapsedTime(&copy_ms, start, stop));

    double total_bytes = 2.0 * (double)bytes * copy_iters;
    double seconds = copy_ms / 1000.0;
    double bandwidth_gbs = total_bytes / seconds / 1e9;

    printf("MEMORY_BANDWIDTH_GBS=%.4f\n", bandwidth_gbs);

    // ---------- Scalar CUDA-core FMA sanity benchmark ----------
    const size_t compute_n = 1ULL << 20;
    const size_t compute_bytes = compute_n * sizeof(float);
    float *c_in = nullptr, *c_out = nullptr;
    CHECK_CUDA(cudaMalloc(&c_in, compute_bytes));
    CHECK_CUDA(cudaMalloc(&c_out, compute_bytes));
    CHECK_CUDA(cudaMemset(c_in, 1, compute_bytes));

    const int loops = 512;
    int c_blocks = (int)((compute_n + threads - 1) / threads);

    for (int i = 0; i < 3; ++i) {
        fma_kernel<<<c_blocks, threads>>>(c_in, c_out, compute_n, loops);
    }
    CHECK_CUDA(cudaDeviceSynchronize());

    CHECK_CUDA(cudaEventRecord(start));
    fma_kernel<<<c_blocks, threads>>>(c_in, c_out, compute_n, loops);
    CHECK_CUDA(cudaEventRecord(stop));
    CHECK_CUDA(cudaEventSynchronize(stop));

    float compute_ms = 0.0f;
    CHECK_CUDA(cudaEventElapsedTime(&compute_ms, start, stop));

    // 8 accumulators * 1 FMA * 2 FLOPs/FMA per loop per thread
    double flops = (double)compute_n * loops * 8.0 * 2.0;
    double scalar_tflops = flops / (compute_ms / 1000.0) / 1e12;

    printf("SCALAR_FMA_TFLOPS=%.4f\n", scalar_tflops);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_in);
    cudaFree(d_out);
    cudaFree(c_in);
    cudaFree(c_out);
    return 0;
}

In [ ]:
# Compile and run the CUDA program.
# -arch=native asks nvcc to target the GPU currently attached to the Colab runtime.
!nvcc -O3 -arch=native gpu_microbench.cu -o gpu_microbench

cuda_output = subprocess.check_output(["./gpu_microbench"], text=True)
print(cuda_output)

def parse_metric(text, key):
    m = re.search(rf"{re.escape(key)}=([0-9.]+)", text)
    if not m:
        raise RuntimeError(f"Could not parse {key}")
    return float(m.group(1))

MEASURED_BW_GBS = parse_metric(cuda_output, "MEMORY_BANDWIDTH_GBS")
SCALAR_FMA_TFLOPS = parse_metric(cuda_output, "SCALAR_FMA_TFLOPS")

print(f"Measured CUDA memory bandwidth: {MEASURED_BW_GBS:.2f} GB/s")
print(f"Scalar CUDA FMA throughput:      {SCALAR_FMA_TFLOPS:.2f} TFLOP/s")

## 2. Practical FP16 GEMM / Tensor Core compute ceiling

The CUDA scalar-FMA benchmark is useful to prove that we wrote CUDA, but it is not representative of the main LLM kernels.

LLM linear layers are dominated by matrix multiplication, so we measure the practical FP16 GEMM ceiling with PyTorch/cuBLAS and use the **best observed TFLOP/s** as the compute roof.

In [ ]:
def benchmark_fp16_gemm(sizes=(2048, 4096, 8192), warmup=5, iters=10):
    results = []

    for n in sizes:
        try:
            a = torch.randn((n, n), device="cuda", dtype=torch.float16)
            b = torch.randn((n, n), device="cuda", dtype=torch.float16)

            for _ in range(warmup):
                _ = a @ b
            torch.cuda.synchronize()

            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            for _ in range(iters):
                c = a @ b
            end.record()
            torch.cuda.synchronize()

            total_ms = start.elapsed_time(end)
            avg_s = (total_ms / iters) / 1000.0

            # GEMM C = A @ B: approximately 2*N^3 FLOPs.
            flops = 2.0 * (n ** 3)
            tflops = flops / avg_s / 1e12

            results.append({"N": n, "avg_ms": avg_s * 1000, "tflops": tflops})
            print(f"N={n:5d}: {avg_s*1000:8.3f} ms, {tflops:8.2f} TFLOP/s")

            del a, b, c
            torch.cuda.empty_cache()

        except torch.cuda.OutOfMemoryError:
            print(f"N={n}: skipped (OOM)")
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

gemm_df = benchmark_fp16_gemm()
display(gemm_df)

MEASURED_COMPUTE_TFLOPS = gemm_df["tflops"].max()

print()
print(f"Practical FP16 compute roof: {MEASURED_COMPUTE_TFLOPS:.2f} TFLOP/s")
print(f"Practical memory roof:       {MEASURED_BW_GBS:.2f} GB/s")

## 3. Machine balance and bottleneck metric

Machine balance is:

\[
MachineBalance = \frac{PeakCompute}{PeakMemoryBandwidth}
\]

in FLOP/byte.

For each LLM workload:

\[
AI = \frac{EstimatedFLOPs}{EstimatedBytesFromHBM}
\]

Then:

\[
ratio = \frac{AI}{MachineBalance}
\]

and our continuous score is:

\[
score = \tanh(\ln(ratio))
\]

Interpretation:
- **score < -0.20**: memory-bandwidth-bound
- **-0.20 <= score <= +0.20**: balanced
- **score > +0.20**: compute-bound

In [ ]:
PEAK_COMPUTE_FLOPS = MEASURED_COMPUTE_TFLOPS * 1e12
PEAK_BW_BYTES_S = MEASURED_BW_GBS * 1e9
MACHINE_BALANCE = PEAK_COMPUTE_FLOPS / PEAK_BW_BYTES_S

print(f"Machine balance = {MACHINE_BALANCE:.2f} FLOP/byte")

def bottleneck_score(arithmetic_intensity):
    ratio = max(arithmetic_intensity / MACHINE_BALANCE, 1e-12)
    score = math.tanh(math.log(ratio))

    if score < -0.20:
        label = "MEMORY"
    elif score > 0.20:
        label = "COMPUTE"
    else:
        label = "BALANCED"

    return score, label, ratio

## 4. Analytical LLM workload model

This estimates the dominant work of a dense decoder-only Transformer.

### Prefill
- Dense/MLP/linear work: approximately `2 * parameters * batch * sequence_length`
- Attention work: approximately `4 * layers * batch * hidden_size * sequence_length^2`
- HBM bytes: model weights + KV-cache writes

### Decode
- Dense/MLP/linear work: approximately `2 * parameters * batch` per generated token
- Attention work: approximately `4 * layers * batch * hidden_size * context_length`
- HBM bytes: model weights + KV-cache reads/writes

This is intentionally an **analytical performance model**, not a claim that every individual hardware transaction is counted perfectly. It is the metric we validate experimentally with runtime trends.

In [ ]:
def model_stats(model):
    cfg = model.config

    params = sum(p.numel() for p in model.parameters())
    hidden = int(cfg.hidden_size)
    layers = int(cfg.num_hidden_layers)
    heads = int(cfg.num_attention_heads)
    kv_heads = int(getattr(cfg, "num_key_value_heads", heads))
    head_dim = int(getattr(cfg, "head_dim", hidden // heads))

    # The model is loaded in FP16 for the base experiment.
    dtype_bytes = 2

    return {
        "params": params,
        "hidden": hidden,
        "layers": layers,
        "heads": heads,
        "kv_heads": kv_heads,
        "head_dim": head_dim,
        "dtype_bytes": dtype_bytes,
    }


def estimate_work(model, batch, context_len, phase):
    s = model_stats(model)

    p = s["params"]
    h = s["hidden"]
    l = s["layers"]
    kvh = s["kv_heads"]
    hd = s["head_dim"]
    db = s["dtype_bytes"]

    weight_bytes = p * db

    if phase == "prefill":
        # Dense projections / MLPs / output layers, first-order approximation.
        dense_flops = 2.0 * p * batch * context_len

        # QK^T and Attention*V.
        attention_flops = 4.0 * l * batch * h * (context_len ** 2)

        # New K and V tensors written to cache.
        kv_write_bytes = 2.0 * l * batch * context_len * kvh * hd * db

        flops = dense_flops + attention_flops
        bytes_moved = weight_bytes + kv_write_bytes

    elif phase == "decode":
        # One new token per sequence in the batch.
        dense_flops = 2.0 * p * batch

        # New query attends over the existing context.
        attention_flops = 4.0 * l * batch * h * context_len

        # Read existing K,V cache and write the new K,V entry.
        kv_read_bytes = 2.0 * l * batch * context_len * kvh * hd * db
        kv_write_bytes = 2.0 * l * batch * kvh * hd * db

        flops = dense_flops + attention_flops
        bytes_moved = weight_bytes + kv_read_bytes + kv_write_bytes

    else:
        raise ValueError("phase must be 'prefill' or 'decode'")

    arithmetic_intensity = flops / bytes_moved
    score, label, ratio = bottleneck_score(arithmetic_intensity)

    return {
        "est_flops": flops,
        "est_bytes": bytes_moved,
        "arithmetic_intensity": arithmetic_intensity,
        "roofline_ratio": ratio,
        "bottleneck_score": score,
        "bottleneck": label,
    }

## 5. Qwen3 loader and benchmark functions

We use:
- `Qwen/Qwen3-0.6B`
- `Qwen/Qwen3-1.7B`
- `Qwen/Qwen3-4B`

For realistic generation behavior we set `logits_to_keep=1`, because only the newest token's logits are needed. This avoids wasting memory and compute producing vocabulary logits for every prompt token.

In [ ]:
MODEL_IDS = {
    "Qwen3-0.6B": "Qwen/Qwen3-0.6B",
    "Qwen3-1.7B": "Qwen/Qwen3-1.7B",
    "Qwen3-4B": "Qwen/Qwen3-4B",
}

def load_model(model_id):
    torch.cuda.empty_cache()
    gc.collect()

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
    model.eval()

    return tokenizer, model


def timed_prefill(model, batch, context_len, warmup=1, repeats=3):
    vocab = int(model.config.vocab_size)

    # Avoid special assumptions about natural-language tokenization:
    # random valid IDs give a deterministic-shape systems benchmark.
    input_ids = torch.randint(
        low=0,
        high=vocab,
        size=(batch, context_len),
        device="cuda",
        dtype=torch.long,
    )

    with torch.inference_mode():
        for _ in range(warmup):
            _ = model(
                input_ids=input_ids,
                use_cache=True,
                logits_to_keep=1,
            )
        torch.cuda.synchronize()

        times = []
        last_out = None

        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            last_out = model(
                input_ids=input_ids,
                use_cache=True,
                logits_to_keep=1,
            )
            end.record()

            torch.cuda.synchronize()
            times.append(start.elapsed_time(end))

    return float(np.median(times)), last_out


def timed_decode(model, prefill_out, batch, decode_steps=16):
    # Current Transformers Qwen3 uses a Cache object returned as `.cache`.
    cache = prefill_out.cache
    vocab = int(model.config.vocab_size)

    step_times_ms = []

    with torch.inference_mode():
        for _ in range(decode_steps):
            next_ids = torch.randint(
                low=0,
                high=vocab,
                size=(batch, 1),
                device="cuda",
                dtype=torch.long,
            )

            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            out = model(
                input_ids=next_ids,
                cache=cache,
                use_cache=True,
                logits_to_keep=1,
            )
            end.record()

            torch.cuda.synchronize()
            step_times_ms.append(start.elapsed_time(end))
            cache = out.cache

    return float(np.mean(step_times_ms))

## 6. One workload benchmark

Each call records:
- prefill latency
- decode latency per generated token step
- tokens/s
- estimated FLOPs and HBM bytes
- arithmetic intensity
- bottleneck score
- estimated achieved TFLOP/s
- estimated bandwidth utilization

In [ ]:
def make_result_row(model_name, model, batch, context_len, phase, latency_ms):
    work = estimate_work(model, batch, context_len, phase)

    latency_s = latency_ms / 1000.0

    achieved_tflops = work["est_flops"] / latency_s / 1e12
    achieved_bw_gbs = work["est_bytes"] / latency_s / 1e9

    compute_util = achieved_tflops / MEASURED_COMPUTE_TFLOPS
    memory_util = achieved_bw_gbs / MEASURED_BW_GBS

    if phase == "prefill":
        tokens_processed = batch * context_len
    else:
        tokens_processed = batch

    return {
        "gpu": GPU_NAME,
        "model": model_name,
        "params_b": sum(p.numel() for p in model.parameters()) / 1e9,
        "phase": phase,
        "batch": batch,
        "context": context_len,
        "latency_ms": latency_ms,
        "tokens_per_s": tokens_processed / latency_s,
        "est_tflops_achieved": achieved_tflops,
        "est_bw_gbs": achieved_bw_gbs,
        "compute_util_frac": compute_util,
        "memory_util_frac": memory_util,
        **work,
    }


def benchmark_workload(model_name, model, batch, context_len, decode_steps=16):
    rows = []

    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    prefill_ms, prefill_out = timed_prefill(
        model=model,
        batch=batch,
        context_len=context_len,
    )

    rows.append(
        make_result_row(
            model_name=model_name,
            model=model,
            batch=batch,
            context_len=context_len,
            phase="prefill",
            latency_ms=prefill_ms,
        )
    )

    decode_ms = timed_decode(
        model=model,
        prefill_out=prefill_out,
        batch=batch,
        decode_steps=decode_steps,
    )

    # Decode runs after the original context. For the analytical model we use
    # the starting context length; 16 extra tokens only slightly change it.
    rows.append(
        make_result_row(
            model_name=model_name,
            model=model,
            batch=batch,
            context_len=context_len,
            phase="decode",
            latency_ms=decode_ms,
        )
    )

    peak_mem_gib = torch.cuda.max_memory_allocated() / 1024**3
    for row in rows:
        row["peak_allocated_gib"] = peak_mem_gib

    del prefill_out
    torch.cuda.empty_cache()

    return rows

## 7. Experiment plan

Start with `RUN_MODE = "quick"` to make sure everything works.

Then switch to `"full"` for the project dataset.

The 4B model is intentionally tested more conservatively on a 16 GiB T4. Any OOM configuration is caught and skipped instead of killing the whole experiment.

In [ ]:
RUN_MODE = "quick"   # change to "full" after the notebook works end-to-end

if RUN_MODE == "quick":
    EXPERIMENTS = {
        "Qwen3-0.6B": {
            "batches": [1, 4],
            "contexts": [128, 1024],
        },
        "Qwen3-1.7B": {
            "batches": [1, 4],
            "contexts": [128, 1024],
        },
        "Qwen3-4B": {
            "batches": [1],
            "contexts": [128, 1024],
        },
    }
else:
    EXPERIMENTS = {
        "Qwen3-0.6B": {
            "batches": [1, 2, 4, 8],
            "contexts": [128, 512, 1024, 2048, 4096],
        },
        "Qwen3-1.7B": {
            "batches": [1, 2, 4],
            "contexts": [128, 512, 1024, 2048, 4096],
        },
        "Qwen3-4B": {
            "batches": [1, 2],
            "contexts": [128, 512, 1024, 2048],
        },
    }

EXPERIMENTS

In [ ]:
all_rows = []
failures = []

for model_name, cfg in EXPERIMENTS.items():
    model_id = MODEL_IDS[model_name]

    print("\n" + "=" * 90)
    print("Loading:", model_name, model_id)
    print("=" * 90)

    tokenizer, model = load_model(model_id)

    stats = model_stats(model)
    print(
        f"{model_name}: "
        f"{stats['params']/1e9:.3f}B params | "
        f"{stats['layers']} layers | "
        f"hidden={stats['hidden']} | "
        f"KV heads={stats['kv_heads']}"
    )

    for batch in cfg["batches"]:
        for context in cfg["contexts"]:
            print(f"\nBenchmarking {model_name} | batch={batch} | context={context}")

            try:
                rows = benchmark_workload(
                    model_name=model_name,
                    model=model,
                    batch=batch,
                    context_len=context,
                    decode_steps=16,
                )
                all_rows.extend(rows)

                for r in rows:
                    print(
                        f"  {r['phase']:7s} | "
                        f"{r['latency_ms']:8.2f} ms | "
                        f"AI={r['arithmetic_intensity']:8.2f} | "
                        f"score={r['bottleneck_score']:+.3f} | "
                        f"{r['bottleneck']}"
                    )

            except torch.cuda.OutOfMemoryError:
                print("  SKIPPED: CUDA out of memory")
                failures.append({
                    "model": model_name,
                    "batch": batch,
                    "context": context,
                    "reason": "OOM",
                })
                torch.cuda.empty_cache()
                gc.collect()

            except Exception as e:
                print("  SKIPPED:", repr(e))
                failures.append({
                    "model": model_name,
                    "batch": batch,
                    "context": context,
                    "reason": repr(e),
                })
                torch.cuda.empty_cache()
                gc.collect()

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

results = pd.DataFrame(all_rows)
failures_df = pd.DataFrame(failures)

print("\nCompleted rows:", len(results))
print("Failed configurations:", len(failures_df))

## 8. Inspect and export the dataset

In [ ]:
display_cols = [
    "model", "phase", "batch", "context",
    "latency_ms", "tokens_per_s",
    "arithmetic_intensity",
    "bottleneck_score", "bottleneck",
    "est_tflops_achieved", "est_bw_gbs",
    "peak_allocated_gib",
]

display(
    results[display_cols]
    .sort_values(["model", "phase", "batch", "context"])
    .reset_index(drop=True)
)

results.to_csv("llm_roofline_results.csv", index=False)
failures_df.to_csv("llm_roofline_failures.csv", index=False)

hardware_summary = {
    "gpu": GPU_NAME,
    "measured_memory_bandwidth_gbs": MEASURED_BW_GBS,
    "scalar_cuda_fma_tflops": SCALAR_FMA_TFLOPS,
    "measured_fp16_gemm_tflops": MEASURED_COMPUTE_TFLOPS,
    "machine_balance_flop_per_byte": MACHINE_BALANCE,
}
Path("hardware_summary.json").write_text(json.dumps(hardware_summary, indent=2))

print("Saved:")
print(" - llm_roofline_results.csv")
print(" - llm_roofline_failures.csv")
print(" - hardware_summary.json")

## 9. Roofline plot

In [ ]:
def plot_roofline(df):
    if df.empty:
        print("No results to plot.")
        return

    x_min = max(df["arithmetic_intensity"].min() / 3, 1e-2)
    x_max = df["arithmetic_intensity"].max() * 3

    ai = np.logspace(np.log10(x_min), np.log10(x_max), 300)

    memory_roof_tflops = (MEASURED_BW_GBS * 1e9 * ai) / 1e12
    compute_roof_tflops = np.full_like(ai, MEASURED_COMPUTE_TFLOPS)
    roof = np.minimum(memory_roof_tflops, compute_roof_tflops)

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(ai, roof, linewidth=2, label="Measured Roofline")
    ax.axvline(MACHINE_BALANCE, linestyle="--", label="Machine balance")

    markers = {"prefill": "o", "decode": "x"}

    for phase, phase_df in df.groupby("phase"):
        ax.scatter(
            phase_df["arithmetic_intensity"],
            phase_df["est_tflops_achieved"].clip(lower=1e-4),
            marker=markers.get(phase, "o"),
            s=60,
            label=phase,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Arithmetic intensity (FLOP/byte)")
    ax.set_ylabel("Estimated achieved performance (TFLOP/s)")
    ax.set_title(f"LLM Roofline on {GPU_NAME}")
    ax.legend()
    ax.grid(True, which="both", alpha=0.25)

    plt.show()

plot_roofline(results)

## 10. Bottleneck-score plots

Negative = memory bound.  
Positive = compute bound.

In [ ]:
if not results.empty:
    summary = (
        results.groupby(["model", "phase"], as_index=False)["bottleneck_score"]
        .mean()
    )

    labels = [f"{m}\n{p}" for m, p in zip(summary["model"], summary["phase"])]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(labels, summary["bottleneck_score"])
    ax.axhline(0, linewidth=1)
    ax.axhline(-0.20, linestyle="--", linewidth=1)
    ax.axhline(0.20, linestyle="--", linewidth=1)
    ax.set_ylabel("Average bottleneck score")
    ax.set_title("Memory-bound (-) to Compute-bound (+)")
    plt.xticks(rotation=20)
    plt.show()

In [ ]:
def plot_heatmap(df, model_name, phase):
    subset = df[(df["model"] == model_name) & (df["phase"] == phase)].copy()
    if subset.empty:
        print("No data:", model_name, phase)
        return

    pivot = subset.pivot_table(
        index="batch",
        columns="context",
        values="bottleneck_score",
        aggfunc="mean",
    ).sort_index().sort_index(axis=1)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    im = ax.imshow(pivot.values, aspect="auto", vmin=-1, vmax=1)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    ax.set_xlabel("Context length")
    ax.set_ylabel("Batch size")
    ax.set_title(f"{model_name} - {phase} bottleneck score")

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            value = pivot.iloc[i, j]
            if not np.isnan(value):
                ax.text(j, i, f"{value:+.2f}", ha="center", va="center")

    fig.colorbar(im, ax=ax, label="-1 memory   |   +1 compute")
    plt.show()

for model_name in MODEL_IDS:
    plot_heatmap(results, model_name, "prefill")
    plot_heatmap(results, model_name, "decode")

## 11. Useful summary tables for the report

In [ ]:
if not results.empty:
    report_summary = (
        results.groupby(["model", "phase"])
        .agg(
            workloads=("bottleneck_score", "count"),
            avg_latency_ms=("latency_ms", "mean"),
            avg_ai=("arithmetic_intensity", "mean"),
            avg_score=("bottleneck_score", "mean"),
            avg_tokens_s=("tokens_per_s", "mean"),
        )
        .reset_index()
    )

    display(report_summary)

    class_counts = (
        results.groupby(["model", "phase", "bottleneck"])
        .size()
        .reset_index(name="count")
    )

    display(class_counts)

# Phase 2 — Quantization validation, recommendation engine, and arbitrary-model analysis

The baseline experiment above established a clear pattern on the T4:

- **Decode** has very low arithmetic intensity and is strongly memory-dominated.
- **Prefill** moves toward / into the compute-bound region as context length and batch size increase.

Phase 2 turns that observation into a usable analyzer.

We now add:

1. **FP16 vs INT8 vs INT4 validation**
   - Does reducing weight memory traffic actually improve a workload predicted to be memory-dominated?
   - Does the same optimization have a different effect on compute-heavy prefill?

2. **Recommendation engine**
   - Converts the measured bottleneck into optimization candidates.
   - Separates *resource dominance* from *actual saturation / under-utilization*.

3. **Generic Hugging Face causal-LM analyzer**
   - Accepts a model ID instead of only the three Qwen models.
   - Supports common dense decoder-only Transformer architectures when their config exposes the required dimensions.
   - Reports limitations instead of claiming unsupported architectures are accurately modeled.

Quantization uses Hugging Face `BitsAndBytesConfig` / bitsandbytes. On a T4, both LLM.int8() and 4-bit NF4/FP4 are supported by bitsandbytes.

In [ ]:
# bitsandbytes is required for 8-bit and 4-bit model loading.
!pip -q install -U bitsandbytes

from transformers import BitsAndBytesConfig, AutoConfig
import inspect

print("bitsandbytes extension ready")

## 12. Generic model/config helpers

The original notebook used Qwen-specific config fields. These helpers recognize the common names used by decoder-only Transformer families such as Qwen, Llama, Mistral, Gemma, GPT-2-style configs, and several others.

This is **not** a promise that every model on Hugging Face is analytically supported. The current FLOP/byte model is designed for **dense decoder-only causal Transformers**. Mixture-of-experts, state-space models, recurrent architectures, multimodal models, and unusual attention mechanisms require architecture-specific accounting.

In [ ]:
def _first_attr(obj, names, default=None):
    for name in names:
        value = getattr(obj, name, None)
        if value is not None:
            return value
    return default


def nominal_param_count(model):
    try:
        return int(model.num_parameters())
    except Exception:
        return int(sum(p.numel() for p in model.parameters()))


def generic_model_stats(model, precision="fp16"):
    cfg = model.config

    if bool(getattr(cfg, "is_encoder_decoder", False)):
        raise ValueError(
            "Current analytical model supports decoder-only causal LMs, not encoder-decoder models."
        )

    hidden = _first_attr(cfg, ["hidden_size", "n_embd", "d_model"])
    layers = _first_attr(cfg, ["num_hidden_layers", "n_layer", "num_layers"])
    heads = _first_attr(cfg, ["num_attention_heads", "n_head", "encoder_attention_heads"])

    if hidden is None or layers is None or heads is None:
        raise ValueError(
            "Could not infer hidden size / layer count / attention heads from this config. "
            "This architecture needs an adapter before the analytical metric is trustworthy."
        )

    hidden = int(hidden)
    layers = int(layers)
    heads = int(heads)

    kv_heads = int(
        _first_attr(cfg, ["num_key_value_heads", "num_kv_heads"], default=heads)
    )
    head_dim = int(_first_attr(cfg, ["head_dim"], default=hidden // heads))

    precision = precision.lower()
    weight_bytes_per_param = {
        "fp16": 2.0,
        "int8": 1.0,
        "int4": 0.5,
    }[precision]

    # Unless a quantized KV cache is explicitly enabled, KV tensors remain
    # approximately FP16 in this experiment.
    kv_bytes_per_element = 2.0

    return {
        "params": nominal_param_count(model),
        "hidden": hidden,
        "layers": layers,
        "heads": heads,
        "kv_heads": kv_heads,
        "head_dim": head_dim,
        "weight_bytes_per_param": weight_bytes_per_param,
        "kv_bytes_per_element": kv_bytes_per_element,
    }


def architecture_notes(config):
    notes = []

    if bool(getattr(config, "is_encoder_decoder", False)):
        notes.append("Encoder-decoder architecture: unsupported by the current analytical model.")

    moe_fields = [
        "num_experts",
        "num_local_experts",
        "n_routed_experts",
        "num_experts_per_tok",
    ]
    if any(getattr(config, field, None) not in (None, 0, 1) for field in moe_fields):
        notes.append(
            "Mixture-of-experts config detected: total parameter count is not equal to active "
            "parameters per token, so the simple FLOP model may overestimate work."
        )

    if getattr(config, "sliding_window", None):
        notes.append(
            "Sliding-window attention detected: long-context KV/attention traffic may be "
            "overestimated by the simple full-context model."
        )

    return notes

## 13. Precision-aware analytical model

For weight-only quantization, the dominant change in our simple memory model is the number of bytes required to fetch model weights:

- FP16: about **2 bytes / parameter**
- INT8: about **1 byte / parameter**
- INT4: about **0.5 byte / parameter**

Real bitsandbytes models have metadata, scales, outlier handling, and kernel overhead, so these values are deliberately treated as an **analytical approximation**. The runtime benchmark is the validation.

In [ ]:
def estimate_work_generic(model, batch, context_len, phase, precision="fp16"):
    s = generic_model_stats(model, precision=precision)

    p = s["params"]
    h = s["hidden"]
    l = s["layers"]
    kvh = s["kv_heads"]
    hd = s["head_dim"]

    weight_bytes = p * s["weight_bytes_per_param"]
    kv_db = s["kv_bytes_per_element"]

    if phase == "prefill":
        dense_flops = 2.0 * p * batch * context_len
        attention_flops = 4.0 * l * batch * h * (context_len ** 2)

        kv_write_bytes = 2.0 * l * batch * context_len * kvh * hd * kv_db

        flops = dense_flops + attention_flops
        bytes_moved = weight_bytes + kv_write_bytes

    elif phase == "decode":
        dense_flops = 2.0 * p * batch
        attention_flops = 4.0 * l * batch * h * context_len

        kv_read_bytes = 2.0 * l * batch * context_len * kvh * hd * kv_db
        kv_write_bytes = 2.0 * l * batch * kvh * hd * kv_db

        flops = dense_flops + attention_flops
        bytes_moved = weight_bytes + kv_read_bytes + kv_write_bytes

    else:
        raise ValueError("phase must be 'prefill' or 'decode'")

    arithmetic_intensity = flops / bytes_moved
    score, label, ratio = bottleneck_score(arithmetic_intensity)

    return {
        "est_flops": flops,
        "est_bytes": bytes_moved,
        "arithmetic_intensity": arithmetic_intensity,
        "roofline_ratio": ratio,
        "bottleneck_score": score,
        "bottleneck": label,
    }

## 14. Generic cache-safe forward helpers

Different Transformers model families expose slightly different forward signatures. The helpers below:

- use `past_key_values` when supported;
- fall back to `cache` for models that expose that API;
- use `logits_to_keep=1` only when the model supports it;
- fail with a clear message if a usable KV cache is not returned.

This keeps the benchmarking path substantially more model-agnostic than the original Qwen-only functions.

In [ ]:
def _forward_param_names(model):
    try:
        return set(inspect.signature(model.forward).parameters.keys())
    except Exception:
        return set()


def _cache_from_output(output):
    cache = getattr(output, "past_key_values", None)
    if cache is None:
        cache = getattr(output, "cache", None)
    return cache


def generic_forward(model, input_ids, cache=None):
    params = _forward_param_names(model)

    kwargs = {
        "input_ids": input_ids,
        "use_cache": True,
    }

    if "logits_to_keep" in params:
        kwargs["logits_to_keep"] = 1
    elif "num_logits_to_keep" in params:
        kwargs["num_logits_to_keep"] = 1

    if cache is not None:
        if "past_key_values" in params:
            kwargs["past_key_values"] = cache
        elif "cache" in params:
            kwargs["cache"] = cache
        else:
            raise ValueError(
                "This model returned a cache, but its forward signature does not expose "
                "past_key_values/cache in a form handled by this notebook."
            )

    return model(**kwargs)


def timed_prefill_generic(model, batch, context_len, warmup=1, repeats=3):
    vocab = int(model.config.vocab_size)

    input_ids = torch.randint(
        0,
        vocab,
        (batch, context_len),
        device="cuda",
        dtype=torch.long,
    )

    with torch.inference_mode():
        for _ in range(warmup):
            _ = generic_forward(model, input_ids)
        torch.cuda.synchronize()

        times = []
        last_out = None

        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            last_out = generic_forward(model, input_ids)
            end.record()
            torch.cuda.synchronize()

            times.append(start.elapsed_time(end))

    cache = _cache_from_output(last_out)
    if cache is None:
        raise ValueError(
            "The model did not return a KV cache with use_cache=True; generic decode "
            "benchmarking is not supported for this model yet."
        )

    return float(np.median(times)), last_out


def timed_decode_generic(model, prefill_out, batch, decode_steps=16):
    cache = _cache_from_output(prefill_out)
    if cache is None:
        raise ValueError("No cache found in prefill output.")

    vocab = int(model.config.vocab_size)
    step_times_ms = []

    with torch.inference_mode():
        for _ in range(decode_steps):
            next_ids = torch.randint(
                0,
                vocab,
                (batch, 1),
                device="cuda",
                dtype=torch.long,
            )

            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)

            start.record()
            out = generic_forward(model, next_ids, cache=cache)
            end.record()
            torch.cuda.synchronize()

            step_times_ms.append(start.elapsed_time(end))

            new_cache = _cache_from_output(out)
            if new_cache is None:
                raise ValueError("Decode step stopped returning a KV cache.")
            cache = new_cache

    return float(np.mean(step_times_ms))

## 15. Load FP16, INT8, or INT4 on the same GPU

For a fair bottleneck experiment we keep the model on the assigned Colab GPU rather than intentionally offloading layers to CPU.

The 4-bit configuration uses NF4 weights and FP16 compute on the T4.

In [ ]:
def load_model_precision(model_id, precision="fp16", trust_remote_code=False):
    precision = precision.lower()

    gc.collect()
    torch.cuda.empty_cache()

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=trust_remote_code)

    common = dict(
        pretrained_model_name_or_path=model_id,
        device_map={"": 0},
        low_cpu_mem_usage=True,
        trust_remote_code=trust_remote_code,
    )

    if precision == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            **common,
            torch_dtype=torch.float16,
        )

    elif precision == "int8":
        qconfig = BitsAndBytesConfig(load_in_8bit=True)

        model = AutoModelForCausalLM.from_pretrained(
            **common,
            quantization_config=qconfig,
            torch_dtype=torch.float16,
        )

    elif precision == "int4":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            **common,
            quantization_config=qconfig,
            torch_dtype=torch.float16,
        )

    else:
        raise ValueError("precision must be one of: fp16, int8, int4")

    model.eval()
    return tokenizer, model


def model_footprint_gib(model):
    try:
        return float(model.get_memory_footprint()) / 1024**3
    except Exception:
        return float("nan")

## 16. Detailed diagnosis + recommendation engine

The original Roofline score answers **which resource the workload demands more heavily**.

That is different from saying the hardware resource is fully saturated.

So the detailed diagnosis now distinguishes:

- **memory-dominated**
- **memory-bandwidth saturated**
- **compute-dominated**
- **compute saturated**
- **balanced**
- **under-utilized / latency- or kernel-overhead-limited**

The recommendation engine returns *optimization candidates*, not guarantees. Phase 2 then tests those candidates experimentally.

In [ ]:
def detailed_diagnosis(row):
    base = row["bottleneck"]
    cutil = float(row.get("compute_util_frac", np.nan))
    mutil = float(row.get("memory_util_frac", np.nan))

    both_known = np.isfinite(cutil) and np.isfinite(mutil)

    if both_known and cutil < 0.35 and mutil < 0.35:
        if base == "MEMORY":
            return "MEMORY-DOMINATED + UNDERUTILIZED/LATENCY-LIMITED"
        if base == "COMPUTE":
            return "COMPUTE-DOMINATED + UNDERUTILIZED/SHAPE-LIMITED"
        return "BALANCED BUT UNDERUTILIZED"

    if base == "MEMORY":
        if both_known and mutil >= 0.70:
            return "MEMORY-BANDWIDTH SATURATED"
        return "MEMORY-DOMINATED"

    if base == "COMPUTE":
        if both_known and cutil >= 0.70:
            return "COMPUTE SATURATED"
        return "COMPUTE-DOMINATED"

    return "BALANCED"


def recommend_optimizations(row):
    diagnosis = detailed_diagnosis(row)
    phase = str(row.get("phase", "")).lower()
    recs = []

    if "MEMORY" in diagnosis:
        recs.extend([
            "Test INT8 or INT4 weight quantization to reduce model-weight traffic.",
            "Increase batch size when the latency budget allows, improving weight reuse.",
        ])

        if phase == "decode":
            recs.append(
                "For long-context decode, test KV-cache reduction/quantization because cache traffic grows with context."
            )

        recs.append(
            "Use fused/optimized inference kernels to reduce unnecessary memory movement."
        )

    elif "COMPUTE" in diagnosis:
        recs.extend([
            "Keep Tensor-Core-friendly precision (FP16/BF16 where supported).",
            "Use optimized attention/GEMM kernels and kernel fusion.",
            "Test torch.compile / compilation where the model and cache path support it.",
        ])

    else:
        recs.extend([
            "Optimize both memory traffic and compute kernels; the workload is near the Roofline crossover.",
            "Benchmark batching and lower precision to see which direction moves the workload."
        ])

    if "UNDERUTILIZED" in diagnosis:
        recs.extend([
            "Increase batch/work size before assuming the GPU needs more bandwidth or compute.",
            "Measure kernel-launch overhead and consider CUDA Graphs / compilation for repeated decode steps."
        ])

    return recs

## 17. Benchmark one model at one precision

This version records both the analytical prediction and the measured runtime behavior.

In [ ]:
def make_generic_result_row(
    model_id,
    model,
    precision,
    batch,
    context_len,
    phase,
    latency_ms,
    footprint_gib,
):
    work = estimate_work_generic(
        model=model,
        batch=batch,
        context_len=context_len,
        phase=phase,
        precision=precision,
    )

    latency_s = latency_ms / 1000.0

    achieved_tflops = work["est_flops"] / latency_s / 1e12
    achieved_bw_gbs = work["est_bytes"] / latency_s / 1e9

    compute_util = achieved_tflops / MEASURED_COMPUTE_TFLOPS
    memory_util = achieved_bw_gbs / MEASURED_BW_GBS

    tokens_processed = batch * context_len if phase == "prefill" else batch

    row = {
        "gpu": GPU_NAME,
        "model_id": model_id,
        "precision": precision,
        "params_b": nominal_param_count(model) / 1e9,
        "phase": phase,
        "batch": batch,
        "context": context_len,
        "latency_ms": latency_ms,
        "tokens_per_s": tokens_processed / latency_s,
        "model_footprint_gib": footprint_gib,
        "est_tflops_achieved": achieved_tflops,
        "est_bw_gbs": achieved_bw_gbs,
        "compute_util_frac": compute_util,
        "memory_util_frac": memory_util,
        **work,
    }

    row["detailed_diagnosis"] = detailed_diagnosis(row)
    return row


def benchmark_loaded_model_generic(
    model_id,
    model,
    precision="fp16",
    batch=1,
    context_len=1024,
    decode_steps=16,
):
    rows = []

    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    footprint_gib = model_footprint_gib(model)

    prefill_ms, prefill_out = timed_prefill_generic(
        model=model,
        batch=batch,
        context_len=context_len,
    )

    rows.append(
        make_generic_result_row(
            model_id=model_id,
            model=model,
            precision=precision,
            batch=batch,
            context_len=context_len,
            phase="prefill",
            latency_ms=prefill_ms,
            footprint_gib=footprint_gib,
        )
    )

    decode_ms = timed_decode_generic(
        model=model,
        prefill_out=prefill_out,
        batch=batch,
        decode_steps=decode_steps,
    )

    rows.append(
        make_generic_result_row(
            model_id=model_id,
            model=model,
            precision=precision,
            batch=batch,
            context_len=context_len,
            phase="decode",
            latency_ms=decode_ms,
            footprint_gib=footprint_gib,
        )
    )

    peak_gib = torch.cuda.max_memory_allocated() / 1024**3
    for row in rows:
        row["peak_allocated_gib"] = peak_gib

    del prefill_out
    torch.cuda.empty_cache()

    return rows

# 18. Quantization validation experiment

We now directly test the recommendation generated by Phase 1.

**Chosen workload:** `Qwen/Qwen3-1.7B`, batch 1, context 1024.

The baseline results above predict:
- decode: strongly memory-dominated;
- prefill: strongly compute-dominated.

We run the same workload with FP16, INT8, and INT4.

Important: lower-bit storage does **not** guarantee lower latency. Quantization/dequantization kernels can add overhead, especially on small models or small batch sizes. The experiment records the real result either way.

In [ ]:
QUANT_MODEL_ID = "Qwen/Qwen3-1.7B"
QUANT_BATCH = 1
QUANT_CONTEXT = 1024
QUANT_PRECISIONS = ["fp16", "int8", "int4"]

quant_rows = []
quant_failures = []

for precision in QUANT_PRECISIONS:
    print("\n" + "=" * 88)
    print(f"{QUANT_MODEL_ID} | {precision.upper()} | batch={QUANT_BATCH} | context={QUANT_CONTEXT}")
    print("=" * 88)

    try:
        tokenizer_q, model_q = load_model_precision(
            QUANT_MODEL_ID,
            precision=precision,
        )

        notes = architecture_notes(model_q.config)
        for note in notes:
            print("NOTE:", note)

        rows = benchmark_loaded_model_generic(
            model_id=QUANT_MODEL_ID,
            model=model_q,
            precision=precision,
            batch=QUANT_BATCH,
            context_len=QUANT_CONTEXT,
            decode_steps=16,
        )

        quant_rows.extend(rows)

        for r in rows:
            print(
                f"{r['phase']:7s} | "
                f"{r['latency_ms']:8.2f} ms | "
                f"{r['tokens_per_s']:8.2f} tok/s | "
                f"AI={r['arithmetic_intensity']:8.2f} | "
                f"score={r['bottleneck_score']:+.3f} | "
                f"{r['detailed_diagnosis']}"
            )

        print(f"Model footprint: {model_footprint_gib(model_q):.3f} GiB")

        del model_q, tokenizer_q
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print("FAILED:", repr(e))
        quant_failures.append({
            "precision": precision,
            "reason": repr(e),
        })
        gc.collect()
        torch.cuda.empty_cache()

quant_results = pd.DataFrame(quant_rows)
quant_failures_df = pd.DataFrame(quant_failures)

display(quant_results)
if not quant_failures_df.empty:
    display(quant_failures_df)

## 19. Did quantization validate the memory-bottleneck recommendation?

For each phase we compute speedup relative to FP16:

\[
speedup = \frac{FP16\ latency}{quantized\ latency}
\]

- `speedup > 1`: faster than FP16
- `speedup < 1`: slower than FP16

This avoids confirmation bias: the notebook explicitly reports when memory compression does **not** translate into runtime speedup.

In [ ]:
def add_speedups(df):
    if df.empty:
        return df.copy()

    out = df.copy()
    out["speedup_vs_fp16"] = np.nan
    out["memory_reduction_vs_fp16_pct"] = np.nan

    for phase in out["phase"].unique():
        phase_mask = out["phase"] == phase
        phase_df = out[phase_mask]

        fp16 = phase_df[phase_df["precision"] == "fp16"]
        if fp16.empty:
            continue

        fp16_latency = float(fp16.iloc[0]["latency_ms"])
        fp16_footprint = float(fp16.iloc[0]["model_footprint_gib"])

        out.loc[phase_mask, "speedup_vs_fp16"] = (
            fp16_latency / out.loc[phase_mask, "latency_ms"]
        )

        if np.isfinite(fp16_footprint) and fp16_footprint > 0:
            out.loc[phase_mask, "memory_reduction_vs_fp16_pct"] = (
                1.0
                - out.loc[phase_mask, "model_footprint_gib"] / fp16_footprint
            ) * 100.0

    return out


quant_results = add_speedups(quant_results)

quant_summary_cols = [
    "precision",
    "phase",
    "latency_ms",
    "tokens_per_s",
    "model_footprint_gib",
    "memory_reduction_vs_fp16_pct",
    "arithmetic_intensity",
    "bottleneck_score",
    "detailed_diagnosis",
    "speedup_vs_fp16",
]

display(
    quant_results[quant_summary_cols]
    .sort_values(["phase", "precision"])
    .reset_index(drop=True)
)

In [ ]:
def interpret_quantization_validation(df):
    if df.empty:
        print("No quantization results available.")
        return

    print("=" * 78)
    print("QUANTIZATION VALIDATION")
    print("=" * 78)

    for phase in ["decode", "prefill"]:
        part = df[df["phase"] == phase].copy()
        if part.empty:
            continue

        fp16 = part[part["precision"] == "fp16"]
        if fp16.empty:
            continue

        baseline_diag = fp16.iloc[0]["detailed_diagnosis"]

        print(f"\n{phase.upper()}")
        print("Baseline diagnosis:", baseline_diag)

        tested = part[part["precision"] != "fp16"]
        if tested.empty:
            print("No quantized comparison completed.")
            continue

        best = tested.loc[tested["speedup_vs_fp16"].idxmax()]
        speedup = float(best["speedup_vs_fp16"])

        print(
            f"Best quantized runtime: {best['precision'].upper()} "
            f"at {speedup:.3f}x FP16 speed"
        )

        if speedup >= 1.05:
            print(
                "Result: runtime improved by at least 5%. "
                "This supports the optimization recommendation for this workload."
            )
        elif speedup >= 0.95:
            print(
                "Result: runtime is within about 5% of FP16. "
                "Memory footprint improved, but there is no clear latency win."
            )
        else:
            print(
                "Result: quantization reduced storage but made this workload slower. "
                "Likely causes include quantization/dequantization kernel overhead, "
                "small GEMM shapes, or low GPU utilization."
            )

interpret_quantization_validation(quant_results)

In [ ]:
if not quant_results.empty:
    latency_pivot = (
        quant_results.pivot_table(
            index="precision",
            columns="phase",
            values="latency_ms",
            aggfunc="mean",
        )
        .reindex(["fp16", "int8", "int4"])
    )

    ax = latency_pivot.plot(
        kind="bar",
        figsize=(9, 5),
        rot=0,
    )
    ax.set_title("Qwen3-1.7B: FP16 vs INT8 vs INT4 latency")
    ax.set_ylabel("Latency (ms)")
    ax.set_xlabel("Precision")
    plt.show()

    footprint = (
        quant_results[["precision", "model_footprint_gib"]]
        .drop_duplicates()
        .set_index("precision")
        .reindex(["fp16", "int8", "int4"])
    )

    ax = footprint.plot(
        kind="bar",
        figsize=(8, 4.5),
        legend=False,
        rot=0,
    )
    ax.set_title("Model memory footprint by precision")
    ax.set_ylabel("GiB")
    ax.set_xlabel("Precision")
    plt.show()

# 20. Automatic performance report

This converts raw metrics into the kind of output the final project should expose to a user.

In [ ]:
def print_performance_report(df):
    if df.empty:
        print("No benchmark rows.")
        return

    first = df.iloc[0]

    print("=" * 88)
    print("LLM GPU DIAGNOSTIC")
    print("=" * 88)
    print("GPU:       ", first["gpu"])
    print("Model:     ", first.get("model_id", first.get("model", "unknown")))
    print("Precision: ", str(first.get("precision", "fp16")).upper())
    print("Batch:     ", int(first["batch"]))
    print("Context:   ", int(first["context"]))

    for phase in ["prefill", "decode"]:
        part = df[df["phase"] == phase]
        if part.empty:
            continue

        row = part.iloc[0]
        diagnosis = detailed_diagnosis(row)

        print("\n" + "-" * 88)
        print(phase.upper())
        print("-" * 88)
        print(f"Latency:               {row['latency_ms']:.3f} ms")
        print(f"Throughput:            {row['tokens_per_s']:.3f} tokens/s")
        print(f"Arithmetic intensity:  {row['arithmetic_intensity']:.3f} FLOP/byte")
        print(f"Bottleneck score:      {row['bottleneck_score']:+.3f}")
        print(f"Est. compute use:      {100*row['compute_util_frac']:.1f}% of measured roof")
        print(f"Est. bandwidth use:    {100*row['memory_util_frac']:.1f}% of measured roof")
        print("Diagnosis:             ", diagnosis)

        print("\nOptimization candidates:")
        for i, rec in enumerate(recommend_optimizations(row), start=1):
            print(f"  {i}. {rec}")

    print("\n" + "=" * 88)
    print(
        "Recommendations are hypotheses generated from the measured/analytical bottleneck. "
        "Validate them with an A/B benchmark before treating them as performance wins."
    )
    print("=" * 88)


# Example: print reports for the quantization runs.
for precision in ["fp16", "int8", "int4"]:
    part = quant_results[quant_results["precision"] == precision]
    if not part.empty:
        print_performance_report(part)
        print()

# 21. Arbitrary supported Hugging Face model analyzer

This is the user-facing direction of the project.

Set a Hugging Face causal-LM model ID below and the notebook will:

1. inspect the architecture;
2. load the requested precision;
3. benchmark prefill and decode;
4. estimate FLOPs and memory traffic;
5. classify the bottleneck;
6. detect likely under-utilization;
7. produce optimization recommendations.

The current analytical model is best suited to **dense decoder-only Transformer LMs**. A model loading successfully does not automatically mean its architecture is perfectly represented by the simple FLOP/byte equations.

In [ ]:
def analyze_huggingface_model(
    model_id,
    batch=1,
    context_len=1024,
    precision="fp16",
    decode_steps=16,
    trust_remote_code=False,
):
    print("Inspecting config:", model_id)

    config = AutoConfig.from_pretrained(model_id, trust_remote_code=trust_remote_code)
    notes = architecture_notes(config)

    if notes:
        print("\nArchitecture notes:")
        for note in notes:
            print(" -", note)

    if bool(getattr(config, "is_encoder_decoder", False)):
        raise ValueError(
            "This notebook currently supports decoder-only causal LMs for the analytical metric."
        )

    tokenizer, model = load_model_precision(
        model_id=model_id,
        precision=precision,
        trust_remote_code=trust_remote_code,
    )

    try:
        stats = generic_model_stats(model, precision=precision)

        print("\nDetected architecture:")
        print(f" Parameters:       {stats['params']/1e9:.3f}B")
        print(f" Layers:           {stats['layers']}")
        print(f" Hidden size:      {stats['hidden']}")
        print(f" Attention heads:  {stats['heads']}")
        print(f" KV heads:         {stats['kv_heads']}")
        print(f" Head dim:         {stats['head_dim']}")
        print(f" Model footprint:  {model_footprint_gib(model):.3f} GiB")

        rows = benchmark_loaded_model_generic(
            model_id=model_id,
            model=model,
            precision=precision,
            batch=batch,
            context_len=context_len,
            decode_steps=decode_steps,
        )

        report = pd.DataFrame(rows)
        print()
        print_performance_report(report)

        return report

    finally:
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()

### Try another model

Start with an openly accessible decoder-only model that fits the T4.

Examples you can try include another Qwen model or another compatible Hugging Face causal LM. Some repositories are gated and require a Hugging Face token/license acceptance before Colab can download them.

In [ ]:
# Change only these values to analyze another supported model.

ARBITRARY_MODEL_ID = "Qwen/Qwen3-0.6B"
ARBITRARY_BATCH = 1
ARBITRARY_CONTEXT = 1024
ARBITRARY_PRECISION = "fp16"   # "fp16", "int8", or "int4"

arbitrary_report = analyze_huggingface_model(
    model_id=ARBITRARY_MODEL_ID,
    batch=ARBITRARY_BATCH,
    context_len=ARBITRARY_CONTEXT,
    precision=ARBITRARY_PRECISION,
    decode_steps=16,
)

display(arbitrary_report)

# 22. Export the Phase 2 results

These files can be used directly for the final report, plots, or a future Streamlit dashboard.

In [ ]:
quant_results.to_csv("quantization_validation_results.csv", index=False)

if "arbitrary_report" in globals() and arbitrary_report is not None:
    arbitrary_report.to_csv("arbitrary_model_report.csv", index=False)

print("Saved:")
print(" - quantization_validation_results.csv")
if "arbitrary_report" in globals() and arbitrary_report is not None:
    print(" - arbitrary_model_report.csv")

# Phase 3 — Cross-model validation

The controlled Qwen family tells us how the metric behaves as model size changes while the architecture family stays similar. Now we test whether the **same analyzer pipeline** works on non-Qwen decoder-only Transformers.

We use:

- **TinyLlama/TinyLlama-1.1B-Chat-v1.0** in FP16
- **microsoft/Phi-3-mini-4k-instruct** in INT4 to keep the larger 3.8B model practical on a 16 GB T4

This phase is about **generalization of the analyzer**, not model quality.

If a model is unsupported, the notebook records the failure and continues instead of silently returning a questionable diagnosis.

In [ ]:
CROSS_MODEL_TESTS = [
    {
        "name": "TinyLlama-b1",
        "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "batch": 1,
        "context": 1024,
        "precision": "fp16",
        "decode_steps": 16,
        "trust_remote_code": False,
    },
    {
        "name": "TinyLlama-b4",
        "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "batch": 4,
        "context": 1024,
        "precision": "fp16",
        "decode_steps": 16,
        "trust_remote_code": False,
    },
    {
        "name": "Phi3-mini-int4",
        "model_id": "microsoft/Phi-3-mini-4k-instruct",
        "batch": 1,
        "context": 1024,
        "precision": "int4",
        "decode_steps": 16,
        "trust_remote_code": True,
    },
]

RUN_CROSS_MODEL_VALIDATION = True
CROSS_MODEL_TESTS

In [ ]:
cross_model_reports = []
cross_model_failures = []

if RUN_CROSS_MODEL_VALIDATION:
    for spec in CROSS_MODEL_TESTS:
        print("\n" + "=" * 92)
        print("CROSS-MODEL TEST:", spec["name"])
        print("=" * 92)

        try:
            report = analyze_huggingface_model(
                model_id=spec["model_id"],
                batch=spec["batch"],
                context_len=spec["context"],
                precision=spec["precision"],
                decode_steps=spec["decode_steps"],
                trust_remote_code=spec["trust_remote_code"],
            )

            report = report.copy()
            report["test_name"] = spec["name"]
            cross_model_reports.append(report)

        except Exception as e:
            print("SKIPPED:", repr(e))
            cross_model_failures.append({
                "test_name": spec["name"],
                "model_id": spec["model_id"],
                "reason": repr(e),
            })
            gc.collect()
            torch.cuda.empty_cache()

if cross_model_reports:
    cross_model_results = pd.concat(cross_model_reports, ignore_index=True)
else:
    cross_model_results = pd.DataFrame()

cross_model_failures_df = pd.DataFrame(cross_model_failures)

if not cross_model_results.empty:
    display(
        cross_model_results[
            [
                "test_name", "model_id", "precision", "phase", "batch", "context",
                "latency_ms", "tokens_per_s", "arithmetic_intensity",
                "bottleneck_score", "detailed_diagnosis", "model_footprint_gib"
            ]
        ]
    )

if not cross_model_failures_df.empty:
    print("Unsupported/failed cross-model tests:")
    display(cross_model_failures_df)

In [ ]:
if not cross_model_results.empty:
    plot_df = cross_model_results.copy()
    plot_df["label"] = (
        plot_df["test_name"].astype(str)
        + " | "
        + plot_df["phase"].astype(str)
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(plot_df["label"], plot_df["bottleneck_score"])
    ax.axhline(0, linewidth=1)
    ax.axhline(-0.20, linestyle="--", linewidth=1)
    ax.axhline(0.20, linestyle="--", linewidth=1)
    ax.set_ylabel("Bottleneck score")
    ax.set_title("Cross-model validation: memory (-) to compute (+)")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

## What cross-model validation is checking

A successful result does **not** mean every model has identical performance. It means the same hardware-aware pipeline can inspect a different decoder-only Transformer, benchmark its prefill/decode phases, compute the Roofline quantities, and return a diagnosis using the same metric.

A model-specific failure is also useful: it identifies an architecture that needs a dedicated FLOP/memory adapter instead of pretending the generic equations are universal.

# Phase 4 — Final user-facing arbitrary-model analyzer

This is the product-like endpoint of the project.

A user provides:

- Hugging Face model ID
- batch size
- context length
- precision
- number of decode steps

The analyzer returns:

- detected architecture statistics
- latency and throughput
- estimated FLOPs and bytes moved
- arithmetic intensity
- compute and bandwidth utilization relative to the measured GPU roofs
- bottleneck score
- detailed diagnosis
- optimization recommendations

The current analytical model is intended for **dense decoder-only Transformer causal LMs**. MoE, encoder-decoder, state-space, multimodal, and unusual attention architectures may require adapters.

In [ ]:
def plot_diagnostic_report(report):
    if report is None or report.empty:
        print("No report to plot.")
        return

    view = report.copy()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(view["phase"], view["bottleneck_score"])
    ax.axhline(0, linewidth=1)
    ax.axhline(-0.20, linestyle="--", linewidth=1)
    ax.axhline(0.20, linestyle="--", linewidth=1)
    ax.set_ylim(-1.05, 1.05)
    ax.set_ylabel("Bottleneck score")
    ax.set_title("Memory-dominated (-) vs compute-dominated (+)")
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(view))
    width = 0.35
    ax.bar(x - width/2, 100 * view["compute_util_frac"], width, label="Compute roof use")
    ax.bar(x + width/2, 100 * view["memory_util_frac"], width, label="Bandwidth roof use")
    ax.set_xticks(x)
    ax.set_xticklabels(view["phase"])
    ax.set_ylabel("Estimated utilization (%)")
    ax.set_title("Estimated use of measured GPU roofs")
    ax.legend()
    plt.show()

In [ ]:
def run_final_analyzer(
    model_id,
    batch=1,
    context_len=1024,
    precision="fp16",
    decode_steps=16,
    trust_remote_code=False,
    save_prefix="final_llm_diagnostic",
):
    report = analyze_huggingface_model(
        model_id=model_id,
        batch=batch,
        context_len=context_len,
        precision=precision,
        decode_steps=decode_steps,
        trust_remote_code=trust_remote_code,
    )

    plot_diagnostic_report(report)

    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_id)
    csv_path = f"{save_prefix}_{safe_name}.csv"
    json_path = f"{save_prefix}_{safe_name}.json"

    report.to_csv(csv_path, index=False)
    Path(json_path).write_text(
        json.dumps(report.to_dict(orient="records"), indent=2, default=str)
    )

    print("Saved:")
    print(" -", csv_path)
    print(" -", json_path)

    return report

## Optional Colab form

The form below wraps the same analyzer in a simple interface. If widgets do not render in a particular runtime, use `run_final_analyzer(...)` directly in the next cell instead.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

model_widget = widgets.Text(
    value="Qwen/Qwen3-0.6B",
    description="Model:",
    layout=widgets.Layout(width="700px"),
)

batch_widget = widgets.Dropdown(
    options=[1, 2, 4, 8],
    value=1,
    description="Batch:",
)

context_widget = widgets.Dropdown(
    options=[128, 512, 1024, 2048, 4096],
    value=1024,
    description="Context:",
)

precision_widget = widgets.Dropdown(
    options=["fp16", "int8", "int4"],
    value="fp16",
    description="Precision:",
)

decode_widget = widgets.IntSlider(
    value=16,
    min=4,
    max=64,
    step=4,
    description="Decode steps:",
)

trust_widget = widgets.Checkbox(
    value=False,
    description="trust_remote_code",
)

run_button = widgets.Button(
    description="Analyze LLM",
    button_style="success",
)

ui_output = widgets.Output()


def _run_from_widget(_):
    with ui_output:
        clear_output(wait=True)
        try:
            global final_interactive_report
            final_interactive_report = run_final_analyzer(
                model_id=model_widget.value.strip(),
                batch=int(batch_widget.value),
                context_len=int(context_widget.value),
                precision=precision_widget.value,
                decode_steps=int(decode_widget.value),
                trust_remote_code=bool(trust_widget.value),
            )
        except Exception as e:
            print("Analyzer failed:", repr(e))
            print(
                "If this is an unusual architecture, it may need a dedicated analytical adapter. "
                "If it is a gated model, authenticate with Hugging Face first."
            )


run_button.on_click(_run_from_widget)

display(
    widgets.VBox([
        model_widget,
        widgets.HBox([batch_widget, context_widget, precision_widget]),
        widgets.HBox([decode_widget, trust_widget]),
        run_button,
        ui_output,
    ])
)

## Direct-call alternative

Edit these values and run the cell when you want a reproducible final demo without the widget UI.

In [ ]:
FINAL_MODEL_ID = "Qwen/Qwen3-0.6B"
FINAL_BATCH = 1
FINAL_CONTEXT = 1024
FINAL_PRECISION = "fp16"
FINAL_DECODE_STEPS = 16

# Uncomment to run a direct final demo:
# final_report = run_final_analyzer(
#     model_id=FINAL_MODEL_ID,
#     batch=FINAL_BATCH,
#     context_len=FINAL_CONTEXT,
#     precision=FINAL_PRECISION,
#     decode_steps=FINAL_DECODE_STEPS,
# )

# Phase 5 — Consolidated project exports

This cell saves the main datasets produced by the notebook. It only exports objects that were actually created in the current runtime.

In [ ]:
exported = []

if "results" in globals() and isinstance(results, pd.DataFrame) and not results.empty:
    results.to_csv("phase1_qwen_scaling_results.csv", index=False)
    exported.append("phase1_qwen_scaling_results.csv")

if "quant_results" in globals() and isinstance(quant_results, pd.DataFrame) and not quant_results.empty:
    quant_results.to_csv("phase2_quantization_validation.csv", index=False)
    exported.append("phase2_quantization_validation.csv")

if "cross_model_results" in globals() and isinstance(cross_model_results, pd.DataFrame) and not cross_model_results.empty:
    cross_model_results.to_csv("phase3_cross_model_validation.csv", index=False)
    exported.append("phase3_cross_model_validation.csv")

if "cross_model_failures_df" in globals() and isinstance(cross_model_failures_df, pd.DataFrame) and not cross_model_failures_df.empty:
    cross_model_failures_df.to_csv("phase3_cross_model_failures.csv", index=False)
    exported.append("phase3_cross_model_failures.csv")

summary = {
    "gpu": GPU_NAME,
    "measured_memory_bandwidth_gbs": float(MEASURED_BW_GBS),
    "measured_fp16_compute_tflops": float(MEASURED_COMPUTE_TFLOPS),
    "machine_balance_flop_per_byte": float(MACHINE_BALANCE),
    "project_claim": (
        "For supported decoder-only Transformer LMs, the analyzer combines measured GPU roofs "
        "with workload arithmetic intensity to diagnose memory-vs-compute dominance, identify "
        "under-utilization, recommend optimizations, and validate selected recommendations empirically."
    ),
}
Path("final_project_hardware_and_claim.json").write_text(json.dumps(summary, indent=2))
exported.append("final_project_hardware_and_claim.json")

print("Exported files:")
for item in exported:
    print(" -", item)

# Final interpretation — what this project is trying to prove

The project is **not** trying to prove that one specific Qwen model is always memory-bound or compute-bound.

The stronger claim is:

> **The bottleneck of LLM inference depends on both the model/workload and the GPU. A hardware-normalized Roofline analyzer can identify whether prefill and decode are primarily memory-dominated or compute-dominated, distinguish resource dominance from simple GPU under-utilization, and use that diagnosis to suggest optimizations that can be tested experimentally.**

### Expected empirical pattern

- Small-batch autoregressive **decode** tends to have low arithmetic intensity because weights and KV data are moved for relatively little computation.
- **Prefill** performs much more computation per weight load; larger context lengths and batches can therefore move it toward the compute roof.
- Increasing **batch size** increases weight reuse and can shift the bottleneck.
- **Quantization** reduces model-weight traffic, but a lower memory footprint does not guarantee lower latency because quantization kernels and small-workload overhead can dominate.
- The **same workload can classify differently on a different GPU** because machine balance depends on that GPU's measured compute and memory roofs.

### Final deliverable

The end product is an **LLM performance doctor**:

`model + GPU + batch + context + precision -> benchmark -> bottleneck diagnosis -> performance statistics -> recommendation -> optional A/B validation`

### Scope statement for the report

The current generic analytical model targets **dense decoder-only Transformer causal LMs**. Architectures such as mixture-of-experts, encoder-decoder Transformers, state-space models, multimodal models, and specialized/sliding attention variants may require architecture-specific FLOP and memory-traffic adapters for high-confidence results.